# AI FinOps - Gestor de Base de Datos SQLite (Usuarios y Roles)

Usa este notebook para consultar, auditar y modificar de manera cómoda los datos de consumo de la base de datos SQLite (`finops_db.sqlite`). Los límites presupuestarios están asociados a los **roles/equipos**, y los usuarios pertenecen a un rol.

In [ ]:
import sqlite3
import os

# Ruta al archivo SQLite de la aplicación
db_path = os.path.join('src', 'app', 'finops_db.sqlite')
print(f"Base de datos: {db_path}")
print(f"Existe el archivo: {'Sí' if os.path.exists(db_path) else 'No'}")

### Función auxiliar para consultas
Esta función ejecuta sentencias SQL. Si es un `SELECT`, devuelve un DataFrame de `pandas` (si está instalado) o una lista de diccionarios.

In [ ]:
def run_query(query, params=()):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    try:
        cursor.execute(query, params)
        if query.strip().upper().startswith("SELECT"):
            rows = cursor.fetchall()
            columns = [col[0] for col in cursor.description]
            data = [dict(r) for r in rows]
            try:
                import pandas as pd
                return pd.DataFrame(data, columns=columns)
            except ImportError:
                return data
        else:
            conn.commit()
            print(f"Operación exitosa. Filas afectadas: {cursor.rowcount}")
    except Exception as e:
        print(f"Error: {e}")
    finally:
        conn.close()

### 1. Consultar Roles (Presupuestos de Equipos)

In [ ]:
run_query("SELECT * FROM roles")

### 2. Consultar Usuarios y su Rol enlazado

In [ ]:
run_query("SELECT u.id as user_id, u.name as user_name, r.name as role_name FROM users u JOIN roles r ON u.role_id = r.id")

### 3. Consultar Historial de Transacciones (Últimas 10)

In [ ]:
run_query("""
    SELECT t.timestamp, u.name as user_name, r.name as role_name, t.model, t.cost 
    FROM transactions t 
    JOIN users u ON t.user_id = u.id 
    JOIN roles r ON u.role_id = r.id 
    ORDER BY t.timestamp DESC 
    LIMIT 10
""")

### 4. Modificar Límites de Presupuesto por Rol
Puedes cambiar el presupuesto asignado a cualquier rol (ej. `marketing`, `producto`) modificando el campo `budget_limit`.

In [ ]:
# Cambiar límite de presupuesto de Marketing a 15.0 dólares
run_query("UPDATE roles SET budget_limit = 15.0 WHERE id = 'marketing'")

# Comprobar el cambio
run_query("SELECT * FROM roles WHERE id = 'marketing'")

### 5. Restablecer el Gasto Acumulado y Alertas de un Rol
Si un rol ha agotado su presupuesto y deseas reiniciarlo:

In [ ]:
target_role = 'marketing'

# Restablecer gasto y bandera de alerta del rol
run_query("UPDATE roles SET spent = 0.0, alert_fired = 0 WHERE id = ?", (target_role,))

# Eliminar alertas asociadas
run_query("DELETE FROM alerts WHERE role_id = ?", (target_role,))

# Verificar
run_query("SELECT * FROM roles WHERE id = ?", (target_role,))